In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

<h1> OLYMPMATH 20 SELECTED BENCHMARK </h1>

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
index_path = "/kaggle/input/oss-benchmark-20/index.csv"
import pandas as pd
index = pd.read_csv(index_path)

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:

    # system_prompt = (
    #     'You are a world-class International Mathematical Olympiad (IMO) competitor. '
    #     'The final answer must be a non-negative integer between 0 and 99999. '
    #     'You must place the final integer answer inside \\boxed{}.'
    # )
    
    # tool_prompt = (
    #     'Use this tool to execute Python code. '
    #     'The environment is a stateful Jupyter notebook. '
    #     'You must use print() to output results.'
    # )
    system_prompt = (
        'You are chatgpt, a large language model trained by openai'
    )

    tool_prompt = (
         'python code execution tool'
    ) #used rude prompt to see if the model performes better or not
    
    preference_prompt = (
        'Use `math`, `numpy`,`sympy`,`itertools` and `collections` to solve the problem.'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/oss-20b-merged-nemotron-low-unmask-think-50k'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17640
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    stream_interval = 200
    context_tokens = 65536
    search_tokens = 1024
    buffer_tokens = 512
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

    def close(self):

        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()

            self._jupyter_session = None

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                # Create stream request with error handling
                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0,deadline - time.time()),
                    )
                except Exception as e:
                    print(f"⚠️ Failed to create completion stream: {e}")
                    # Break this iteration and try next one
                    break
    
                if stream is None:
                    # Stream creation failed, skip this iteration
                    continue

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    print("🐍 Executing Python code...")
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            python_errors += 1

        finally:
            if local_tool is not None:
                local_tool.close()

            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer
        }

    def _select_answer(self, detailed_results: list) -> int:

        stats = defaultdict(lambda: {'votes': 0, 'calls': 0})

        for result in detailed_results:
            answer = result['Answer']

            if answer is not None:
                stats[answer]['votes'] += 1
                stats[answer]['calls'] += result['Python Calls']

        sorted_stats = sorted(
            stats.items(), 
            key=lambda item: (item[1]['votes'], item[1]['calls']), 
            reverse=True
        )

        vote_data = []

        for answer, data in sorted_stats:
            vote_data.append((answer, data['votes'], data['calls']))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Calls'])
        display(vote_dataframe)

        final_answer = sorted_stats[0][0]
        final_votes = sorted_stats[0][1]['votes']
        final_calls = sorted_stats[0][1]['calls']

        print(f'\nFinal Result: {final_answer} | Votes: {final_votes} | Calls: {final_calls}\n')

        # Store for logging
        self._last_majority_votes = final_votes

        return final_answer

    def solve_problem(self, problem: str) -> int:
        
        problem_start_time = time.time()
        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            executor.shutdown(wait=False, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        # print the inference time and budget
        used_time = time.time() - problem_start_time
        saved_time = max(0.0, budget - used_time)
        print(f"[Budget]: {budget:.2f}s\n")
        print(f"[inference] Took {used_time:.2f}s\n")
        print(f"[Saved time]: {saved_time:.2f}s\n")

        # Store detailed results for logging
        self._last_detailed_results = detailed_results
        
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')
            self._last_majority_votes = 0
            return 0

        return self._select_answer(detailed_results)

    def __del__(self):

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
# Load reference data and keep ground truth for accuracy calculation
import json
from datetime import datetime

df = pd.read_csv(
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
)

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Store problem hardness from index.csv (loaded in cell 4)
# index DataFrame has 'id' and 'problem_hardness' columns
problem_hardness = dict(zip(index["id"], index["problem_hardness"])) if "problem_hardness" in index.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

# ============ LOGGING SETUP ============
LOG_DIR = "benchmark_logs"
os.makedirs(LOG_DIR, exist_ok=True)

# Generate timestamp for log files
log_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = os.path.join(LOG_DIR, f"benchmark_log_{log_timestamp}.jsonl")
SUMMARY_FILE = os.path.join(LOG_DIR, f"benchmark_summary_{log_timestamp}.json")

# Global tracking variables for logging
benchmark_start_time = time.time()
all_problem_logs = []

def log_problem_result(
    problem_id,
    question_text,
    final_answer,
    ground_truth_answer,
    is_correct,
    problem_time,
    detailed_results,
    hardness,
    majority_vote_info,
    budget_info
):
    """Log comprehensive information for each problem."""
    
    # Analyze detailed results
    total_attempts = len(detailed_results)
    successful_attempts = sum(1 for r in detailed_results if r.get('Answer') is not None)
    total_python_calls = sum(r.get('Python Calls', 0) for r in detailed_results)
    total_python_errors = sum(r.get('Python Errors', 0) for r in detailed_results)
    total_response_tokens = sum(r.get('Response Length', 0) for r in detailed_results)
    
    # Get all answers and their distribution
    answers = [r.get('Answer') for r in detailed_results if r.get('Answer') is not None]
    answer_distribution = dict(Counter(answers))
    
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "problem_id": problem_id,
        "question_preview": question_text[:500],
        "hardness": hardness,
        "ground_truth": ground_truth_answer,
        "predicted_answer": final_answer,
        "is_correct": is_correct,
        "problem_time_seconds": round(problem_time, 2),
        "budget_seconds": round(budget_info.get("budget", 0), 2),
        "time_saved_seconds": round(budget_info.get("saved", 0), 2),
        "total_attempts": total_attempts,
        "successful_attempts": successful_attempts,
        "failed_attempts": total_attempts - successful_attempts,
        "total_python_calls": total_python_calls,
        "total_python_errors": total_python_errors,
        "python_success_rate": round((total_python_calls - total_python_errors) / max(1, total_python_calls) * 100, 2),
        "total_response_tokens": total_response_tokens,
        "avg_tokens_per_attempt": round(total_response_tokens / max(1, total_attempts), 2),
        "answer_distribution": answer_distribution,
        "majority_vote_answer": majority_vote_info.get("answer"),
        "majority_vote_count": majority_vote_info.get("votes"),
        "majority_vote_confidence": round(majority_vote_info.get("votes", 0) / max(1, successful_attempts) * 100, 2) if successful_attempts > 0 else 0,
        "detailed_results": detailed_results,
    }
    
    # Append to log file
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(log_entry) + "\n")
    
    all_problem_logs.append(log_entry)
    
    return log_entry

def save_benchmark_summary():
    """Save overall benchmark summary."""
    
    total_time = time.time() - benchmark_start_time
    
    # Calculate overall statistics
    total_problems = len(all_problem_logs)
    correct_problems = sum(1 for log in all_problem_logs if log["is_correct"])
    
    # Calculate by hardness
    hard_logs = [log for log in all_problem_logs if log["hardness"] == "HARD"]
    easy_logs = [log for log in all_problem_logs if log["hardness"] == "EASY"]
    unknown_logs = [log for log in all_problem_logs if log["hardness"] not in ["HARD", "EASY"]]
    
    hard_correct = sum(1 for log in hard_logs if log["is_correct"])
    easy_correct = sum(1 for log in easy_logs if log["is_correct"])
    unknown_correct = sum(1 for log in unknown_logs if log["is_correct"])
    
    # Aggregate statistics
    total_python_calls = sum(log["total_python_calls"] for log in all_problem_logs)
    total_python_errors = sum(log["total_python_errors"] for log in all_problem_logs)
    total_tokens = sum(log["total_response_tokens"] for log in all_problem_logs)
    total_attempts = sum(log["total_attempts"] for log in all_problem_logs)
    successful_attempts = sum(log["successful_attempts"] for log in all_problem_logs)
    
    # Time statistics
    problem_times = [log["problem_time_seconds"] for log in all_problem_logs]
    avg_time = sum(problem_times) / max(1, len(problem_times))
    max_time = max(problem_times) if problem_times else 0
    min_time = min(problem_times) if problem_times else 0
    
    # Time by hardness
    hard_times = [log["problem_time_seconds"] for log in hard_logs]
    easy_times = [log["problem_time_seconds"] for log in easy_logs]
    
    summary = {
        "timestamp": datetime.now().isoformat(),
        "log_file": LOG_FILE,
        "total_runtime_seconds": round(total_time, 2),
        "total_runtime_minutes": round(total_time / 60, 2),
        
        "overall_stats": {
            "total_problems": total_problems,
            "correct_problems": correct_problems,
            "accuracy_percent": round(correct_problems / max(1, total_problems) * 100, 2),
        },
        
        "hardness_breakdown": {
            "hard": {
                "total": len(hard_logs),
                "correct": hard_correct,
                "accuracy_percent": round(hard_correct / max(1, len(hard_logs)) * 100, 2) if hard_logs else 0,
                "avg_time_seconds": round(sum(hard_times) / max(1, len(hard_times)), 2) if hard_times else 0,
            },
            "easy": {
                "total": len(easy_logs),
                "correct": easy_correct,
                "accuracy_percent": round(easy_correct / max(1, len(easy_logs)) * 100, 2) if easy_logs else 0,
                "avg_time_seconds": round(sum(easy_times) / max(1, len(easy_times)), 2) if easy_times else 0,
            },
            "unknown": {
                "total": len(unknown_logs),
                "correct": unknown_correct,
                "accuracy_percent": round(unknown_correct / max(1, len(unknown_logs)) * 100, 2) if unknown_logs else 0,
            }
        },
        
        "code_execution_stats": {
            "total_python_calls": total_python_calls,
            "total_python_errors": total_python_errors,
            "python_success_rate": round((total_python_calls - total_python_errors) / max(1, total_python_calls) * 100, 2),
            "avg_python_calls_per_problem": round(total_python_calls / max(1, total_problems), 2),
        },
        
        "inference_stats": {
            "total_attempts": total_attempts,
            "successful_attempts": successful_attempts,
            "attempt_success_rate": round(successful_attempts / max(1, total_attempts) * 100, 2),
            "total_tokens_generated": total_tokens,
            "avg_tokens_per_problem": round(total_tokens / max(1, total_problems), 2),
        },
        
        "time_stats": {
            "avg_time_per_problem_seconds": round(avg_time, 2),
            "max_time_seconds": round(max_time, 2),
            "min_time_seconds": round(min_time, 2),
            "total_budget_used_seconds": round(sum(log["problem_time_seconds"] for log in all_problem_logs), 2),
            "total_time_saved_seconds": round(sum(log["time_saved_seconds"] for log in all_problem_logs), 2),
        },
        
        "problem_details": [
            {
                "id": log["problem_id"],
                "hardness": log["hardness"],
                "correct": log["is_correct"],
                "predicted": log["predicted_answer"],
                "ground_truth": log["ground_truth"],
                "time": log["problem_time_seconds"],
                "python_calls": log["total_python_calls"],
                "majority_confidence": log["majority_vote_confidence"],
            }
            for log in all_problem_logs
        ]
    }
    
    with open(SUMMARY_FILE, "w") as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n📁 Logs saved to: {LOG_FILE}")
    print(f"📊 Summary saved to: {SUMMARY_FILE}")
    
    return summary

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    problem_start = time.time()
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    # Get hardness for this problem
    hardness = problem_hardness.get(question_id, "UNKNOWN")
    print(f"🎯 Hardness: {hardness}")
    
    # Store solver's budget info before solving
    elapsed_global = time.time() - solver.notebook_start_time
    time_left = solver.cfg.notebook_limit - elapsed_global
    problems_left_others = max(0, solver.problems_remaining - 1)
    reserved_time = problems_left_others * solver.cfg.base_problem_timeout
    budget = time_left - reserved_time
    budget = min(budget, solver.cfg.high_problem_timeout)
    budget = max(budget, solver.cfg.base_problem_timeout)
    
    # Solve the problem and capture detailed results
    final_answer = solver.solve_problem(question_text)
    predictions[question_id] = final_answer
    
    problem_time = time.time() - problem_start
    time_saved = max(0, budget - problem_time)
    
    # Get ground truth and check accuracy
    total_count += 1
    gt = ground_truth.get(question_id)
    is_correct = (final_answer == gt) if gt is not None else None
    
    if is_correct:
        correct_count += 1
    
    # Log the result
    log_problem_result(
        problem_id=question_id,
        question_text=question_text,
        final_answer=final_answer,
        ground_truth_answer=gt,
        is_correct=is_correct if is_correct is not None else False,
        problem_time=problem_time,
        detailed_results=getattr(solver, '_last_detailed_results', []),
        hardness=hardness,
        majority_vote_info={
            "answer": final_answer,
            "votes": getattr(solver, '_last_majority_votes', 0),
        },
        budget_info={
            "budget": budget,
            "saved": time_saved,
        }
    )
    
    if gt is not None:
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
        print(f"⏱️ Problem Time: {problem_time:.2f}s | Budget: {budget:.2f}s | Saved: {time_saved:.2f}s")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

inference_server.run_local_gateway(("/kaggle/input/oss-benchmark-20/BENCHMARK.csv",))

# Save benchmark summary after inference completes
benchmark_summary = save_benchmark_summary()
print("\n" + "="*60)
print("BENCHMARK COMPLETED!")
print("="*60)

# 📊 Benchmark Results Analysis

Comprehensive analysis of the benchmark run results from the log files.

In [ ]:
# ============ LOAD AND ANALYZE BENCHMARK RESULTS ============
import json
import glob
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Find the most recent log files
log_files = sorted(glob.glob(os.path.join(LOG_DIR, "benchmark_log_*.jsonl")))
summary_files = sorted(glob.glob(os.path.join(LOG_DIR, "benchmark_summary_*.json")))

if log_files:
    latest_log = log_files[-1]
    print(f"📁 Loading log file: {latest_log}")
    
    # Load all problem logs
    problem_logs = []
    with open(latest_log, "r") as f:
        for line in f:
            problem_logs.append(json.loads(line))
    
    print(f"✅ Loaded {len(problem_logs)} problem records")
else:
    print("⚠️ No log files found. Using in-memory logs.")
    problem_logs = all_problem_logs

if summary_files:
    latest_summary = summary_files[-1]
    print(f"📁 Loading summary file: {latest_summary}")
    
    with open(latest_summary, "r") as f:
        summary_data = json.load(f)
else:
    summary_data = benchmark_summary if 'benchmark_summary' in dir() else None

# Convert to DataFrame for easier analysis
df_results = pd.DataFrame(problem_logs)
print(f"\n📊 Results DataFrame shape: {df_results.shape}")
df_results.head()

In [ ]:
# ============ OVERALL ACCURACY SUMMARY ============
print("=" * 70)
print("📈 OVERALL BENCHMARK SUMMARY")
print("=" * 70)

total_problems = len(df_results)
correct_problems = df_results['is_correct'].sum()
accuracy = correct_problems / total_problems * 100 if total_problems > 0 else 0

print(f"\n🎯 Total Problems: {total_problems}")
print(f"✅ Correct: {correct_problems}")
print(f"❌ Incorrect: {total_problems - correct_problems}")
print(f"📊 Overall Accuracy: {accuracy:.2f}%")

# Time statistics
total_time = df_results['problem_time_seconds'].sum()
avg_time = df_results['problem_time_seconds'].mean()
max_time = df_results['problem_time_seconds'].max()
min_time = df_results['problem_time_seconds'].min()

print(f"\n⏱️ TIME STATISTICS:")
print(f"   Total Time: {total_time:.2f}s ({total_time/60:.2f} min)")
print(f"   Avg Time per Problem: {avg_time:.2f}s")
print(f"   Max Time: {max_time:.2f}s")
print(f"   Min Time: {min_time:.2f}s")
print(f"   Total Time Saved: {df_results['time_saved_seconds'].sum():.2f}s")

# Code execution statistics
total_python_calls = df_results['total_python_calls'].sum()
total_python_errors = df_results['total_python_errors'].sum()
python_success_rate = (total_python_calls - total_python_errors) / total_python_calls * 100 if total_python_calls > 0 else 0

print(f"\n🐍 CODE EXECUTION STATISTICS:")
print(f"   Total Python Calls: {total_python_calls}")
print(f"   Total Python Errors: {total_python_errors}")
print(f"   Python Success Rate: {python_success_rate:.2f}%")
print(f"   Avg Python Calls per Problem: {total_python_calls/total_problems:.2f}")

# Token statistics
total_tokens = df_results['total_response_tokens'].sum()
avg_tokens = df_results['total_response_tokens'].mean()

print(f"\n📝 TOKEN STATISTICS:")
print(f"   Total Tokens Generated: {total_tokens:,}")
print(f"   Avg Tokens per Problem: {avg_tokens:,.0f}")

# Attempt statistics
total_attempts = df_results['total_attempts'].sum()
successful_attempts = df_results['successful_attempts'].sum()
attempt_success_rate = successful_attempts / total_attempts * 100 if total_attempts > 0 else 0

print(f"\n🔄 ATTEMPT STATISTICS:")
print(f"   Total Attempts: {total_attempts}")
print(f"   Successful Attempts: {successful_attempts}")
print(f"   Attempt Success Rate: {attempt_success_rate:.2f}%")

In [ ]:
# ============ HARDNESS ANALYSIS ============
print("=" * 70)
print("🎯 ANALYSIS BY PROBLEM HARDNESS (HARD vs EASY)")
print("=" * 70)

# Separate by hardness
df_hard = df_results[df_results['hardness'] == 'HARD']
df_easy = df_results[df_results['hardness'] == 'EASY']
df_unknown = df_results[~df_results['hardness'].isin(['HARD', 'EASY'])]

# Statistics by hardness
hardness_stats = []

for name, df_subset in [("HARD", df_hard), ("EASY", df_easy), ("UNKNOWN", df_unknown)]:
    if len(df_subset) > 0:
        stats = {
            "Hardness": name,
            "Count": len(df_subset),
            "Correct": df_subset['is_correct'].sum(),
            "Accuracy (%)": round(df_subset['is_correct'].mean() * 100, 2),
            "Avg Time (s)": round(df_subset['problem_time_seconds'].mean(), 2),
            "Avg Python Calls": round(df_subset['total_python_calls'].mean(), 2),
            "Avg Tokens": round(df_subset['total_response_tokens'].mean(), 0),
            "Python Errors": df_subset['total_python_errors'].sum(),
            "Avg Confidence (%)": round(df_subset['majority_vote_confidence'].mean(), 2),
        }
        hardness_stats.append(stats)

df_hardness_stats = pd.DataFrame(hardness_stats)
print("\n📊 Hardness Breakdown:")
display(df_hardness_stats)

# Detailed comparison
print("\n📈 DETAILED HARDNESS COMPARISON:")
if len(df_hard) > 0 and len(df_easy) > 0:
    print(f"\n   HARD Problems ({len(df_hard)} total):")
    print(f"   • Accuracy: {df_hard['is_correct'].mean()*100:.2f}%")
    print(f"   • Avg Time: {df_hard['problem_time_seconds'].mean():.2f}s")
    print(f"   • Avg Python Calls: {df_hard['total_python_calls'].mean():.2f}")
    print(f"   • Total Python Errors: {df_hard['total_python_errors'].sum()}")
    
    print(f"\n   EASY Problems ({len(df_easy)} total):")
    print(f"   • Accuracy: {df_easy['is_correct'].mean()*100:.2f}%")
    print(f"   • Avg Time: {df_easy['problem_time_seconds'].mean():.2f}s")
    print(f"   • Avg Python Calls: {df_easy['total_python_calls'].mean():.2f}")
    print(f"   • Total Python Errors: {df_easy['total_python_errors'].sum()}")
    
    # Calculate difficulty ratio
    hard_acc = df_hard['is_correct'].mean() * 100
    easy_acc = df_easy['is_correct'].mean() * 100
    diff_ratio = (easy_acc - hard_acc) if easy_acc > 0 else 0
    
    print(f"\n   📊 Easy vs Hard Accuracy Gap: {diff_ratio:.2f} percentage points")
    print(f"   📊 HARD/EASY Accuracy Ratio: {hard_acc/easy_acc:.2f}" if easy_acc > 0 else "")

In [ ]:
# ============ VISUALIZATION: PROBLEMS SOLVED BY ID AND HARDNESS ============
print("=" * 70)
print("📊 VISUALIZATION: PROBLEMS SOLVED BY ID AND HARDNESS")
print("=" * 70)

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Sort by problem_id for consistent ordering
df_sorted = df_results.sort_values('problem_id')

# 1. Bar chart showing solved/unsolved by problem ID with hardness coloring
ax1 = axes[0, 0]
problem_ids = df_sorted['problem_id'].tolist()
is_correct_list = df_sorted['is_correct'].tolist()
hardness_list = df_sorted['hardness'].tolist()

# Color mapping: Green for correct, Red for incorrect; different shades for HARD/EASY
colors = []
for correct, hardness in zip(is_correct_list, hardness_list):
    if correct:
        colors.append('#2ECC71' if hardness == 'EASY' else '#27AE60')  # Green shades
    else:
        colors.append('#E74C3C' if hardness == 'HARD' else '#F39C12')  # Red/Orange

bars = ax1.bar(range(len(problem_ids)), [1]*len(problem_ids), color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Problem Index', fontsize=12)
ax1.set_ylabel('Status', fontsize=12)
ax1.set_title('Problems Solved by ID (Green=Correct, Red/Orange=Incorrect)\nDark=HARD, Light=EASY', fontsize=12)
ax1.set_yticks([])

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#27AE60', label='Correct - HARD'),
    Patch(facecolor='#2ECC71', label='Correct - EASY'),
    Patch(facecolor='#E74C3C', label='Incorrect - HARD'),
    Patch(facecolor='#F39C12', label='Incorrect - EASY'),
]
ax1.legend(handles=legend_elements, loc='upper right')

# 2. Pie chart of overall accuracy
ax2 = axes[0, 1]
correct = df_results['is_correct'].sum()
incorrect = len(df_results) - correct
ax2.pie([correct, incorrect], 
        labels=[f'Correct\n({correct})', f'Incorrect\n({incorrect})'],
        colors=['#2ECC71', '#E74C3C'],
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.05, 0))
ax2.set_title('Overall Accuracy', fontsize=14)

# 3. Grouped bar chart by hardness
ax3 = axes[1, 0]
if len(df_hard) > 0 or len(df_easy) > 0:
    categories = []
    correct_counts = []
    incorrect_counts = []
    
    if len(df_hard) > 0:
        categories.append('HARD')
        correct_counts.append(df_hard['is_correct'].sum())
        incorrect_counts.append(len(df_hard) - df_hard['is_correct'].sum())
    
    if len(df_easy) > 0:
        categories.append('EASY')
        correct_counts.append(df_easy['is_correct'].sum())
        incorrect_counts.append(len(df_easy) - df_easy['is_correct'].sum())
    
    x = np.arange(len(categories))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, correct_counts, width, label='Correct', color='#2ECC71')
    bars2 = ax3.bar(x + width/2, incorrect_counts, width, label='Incorrect', color='#E74C3C')
    
    ax3.set_xlabel('Problem Hardness', fontsize=12)
    ax3.set_ylabel('Count', fontsize=12)
    ax3.set_title('Correct vs Incorrect by Hardness Level', fontsize=14)
    ax3.set_xticks(x)
    ax3.set_xticklabels(categories)
    ax3.legend()
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax3.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    for bar in bars2:
        height = bar.get_height()
        ax3.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

# 4. Time distribution by hardness
ax4 = axes[1, 1]
if len(df_hard) > 0 and len(df_easy) > 0:
    data_to_plot = [df_hard['problem_time_seconds'].values, df_easy['problem_time_seconds'].values]
    bp = ax4.boxplot(data_to_plot, labels=['HARD', 'EASY'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#E74C3C')
    bp['boxes'][1].set_facecolor('#2ECC71')
    ax4.set_ylabel('Time (seconds)', fontsize=12)
    ax4.set_title('Problem Solving Time Distribution by Hardness', fontsize=14)
elif len(df_results) > 0:
    ax4.hist(df_results['problem_time_seconds'], bins=20, color='#3498DB', edgecolor='black')
    ax4.set_xlabel('Time (seconds)', fontsize=12)
    ax4.set_ylabel('Count', fontsize=12)
    ax4.set_title('Problem Solving Time Distribution', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'benchmark_visualization_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📁 Visualization saved to: {os.path.join(LOG_DIR, f'benchmark_visualization_{log_timestamp}.png')}")

In [ ]:
# ============ DETAILED PROBLEM-BY-PROBLEM ANALYSIS ============
print("=" * 70)
print("📋 DETAILED PROBLEM-BY-PROBLEM RESULTS")
print("=" * 70)

# Create detailed results table
detailed_table = df_results[[
    'problem_id', 'hardness', 'is_correct', 'predicted_answer', 'ground_truth',
    'problem_time_seconds', 'total_python_calls', 'total_python_errors',
    'successful_attempts', 'total_attempts', 'majority_vote_confidence'
]].copy()

detailed_table.columns = [
    'ID', 'Hardness', 'Correct', 'Predicted', 'Ground Truth',
    'Time (s)', 'Python Calls', 'Python Errors',
    'Success Attempts', 'Total Attempts', 'Vote Confidence (%)'
]

# Add status emoji column
detailed_table.insert(2, 'Status', detailed_table['Correct'].apply(lambda x: '✅' if x else '❌'))

print("\n📊 Full Results Table:")
display(detailed_table)

# Show failed problems
failed_problems = df_results[~df_results['is_correct']]
if len(failed_problems) > 0:
    print(f"\n❌ FAILED PROBLEMS ({len(failed_problems)} total):")
    for _, row in failed_problems.iterrows():
        print(f"   • ID: {row['problem_id']} | Hardness: {row['hardness']} | Predicted: {row['predicted_answer']} | GT: {row['ground_truth']} | Time: {row['problem_time_seconds']:.2f}s")

# Show problems with low confidence
low_confidence = df_results[df_results['majority_vote_confidence'] < 50]
if len(low_confidence) > 0:
    print(f"\n⚠️ LOW CONFIDENCE PROBLEMS (<50% majority vote) ({len(low_confidence)} total):")
    for _, row in low_confidence.iterrows():
        status = "✅" if row['is_correct'] else "❌"
        print(f"   • {status} ID: {row['problem_id']} | Confidence: {row['majority_vote_confidence']:.1f}% | Hardness: {row['hardness']}")

In [ ]:
# ============ CODE EXECUTION ANALYSIS ============
print("=" * 70)
print("🐍 CODE EXECUTION ANALYSIS")
print("=" * 70)

# Create figure for code execution analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Python calls distribution
ax1 = axes[0, 0]
ax1.hist(df_results['total_python_calls'], bins=20, color='#3498DB', edgecolor='black', alpha=0.7)
ax1.axvline(df_results['total_python_calls'].mean(), color='red', linestyle='--', label=f'Mean: {df_results["total_python_calls"].mean():.1f}')
ax1.set_xlabel('Python Calls per Problem', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Python Calls', fontsize=14)
ax1.legend()

# 2. Python errors distribution
ax2 = axes[0, 1]
error_counts = df_results['total_python_errors'].value_counts().sort_index()
ax2.bar(error_counts.index, error_counts.values, color='#E74C3C', edgecolor='black')
ax2.set_xlabel('Python Errors per Problem', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Python Errors', fontsize=14)

# 3. Python calls vs correctness
ax3 = axes[1, 0]
correct_calls = df_results[df_results['is_correct']]['total_python_calls']
incorrect_calls = df_results[~df_results['is_correct']]['total_python_calls']

data = [correct_calls.values, incorrect_calls.values] if len(incorrect_calls) > 0 else [correct_calls.values]
labels = ['Correct', 'Incorrect'] if len(incorrect_calls) > 0 else ['Correct']
colors_box = ['#2ECC71', '#E74C3C'] if len(incorrect_calls) > 0 else ['#2ECC71']

bp = ax3.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax3.set_ylabel('Python Calls', fontsize=12)
ax3.set_title('Python Calls: Correct vs Incorrect Problems', fontsize=14)

# 4. Python success rate by hardness
ax4 = axes[1, 1]
if len(df_hard) > 0 or len(df_easy) > 0:
    categories = []
    success_rates = []
    
    if len(df_hard) > 0:
        hard_calls = df_hard['total_python_calls'].sum()
        hard_errors = df_hard['total_python_errors'].sum()
        hard_rate = (hard_calls - hard_errors) / hard_calls * 100 if hard_calls > 0 else 100
        categories.append('HARD')
        success_rates.append(hard_rate)
    
    if len(df_easy) > 0:
        easy_calls = df_easy['total_python_calls'].sum()
        easy_errors = df_easy['total_python_errors'].sum()
        easy_rate = (easy_calls - easy_errors) / easy_calls * 100 if easy_calls > 0 else 100
        categories.append('EASY')
        success_rates.append(easy_rate)
    
    bars = ax4.bar(categories, success_rates, color=['#E74C3C', '#2ECC71'][:len(categories)], edgecolor='black')
    ax4.set_ylabel('Python Success Rate (%)', fontsize=12)
    ax4.set_title('Python Code Success Rate by Hardness', fontsize=14)
    ax4.set_ylim(0, 105)
    
    for bar, rate in zip(bars, success_rates):
        ax4.annotate(f'{rate:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, rate),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'code_execution_analysis_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print code execution statistics
print("\n📊 CODE EXECUTION STATISTICS BY CORRECTNESS:")
print(f"\n   Correct Problems:")
print(f"   • Avg Python Calls: {df_results[df_results['is_correct']]['total_python_calls'].mean():.2f}")
print(f"   • Avg Python Errors: {df_results[df_results['is_correct']]['total_python_errors'].mean():.2f}")

print(f"\n   Incorrect Problems:")
if len(df_results[~df_results['is_correct']]) > 0:
    print(f"   • Avg Python Calls: {df_results[~df_results['is_correct']]['total_python_calls'].mean():.2f}")
    print(f"   • Avg Python Errors: {df_results[~df_results['is_correct']]['total_python_errors'].mean():.2f}")
else:
    print(f"   • No incorrect problems!")

In [ ]:
# ============ MAJORITY VOTE & CONFIDENCE ANALYSIS ============
print("=" * 70)
print("🗳️ MAJORITY VOTE & CONFIDENCE ANALYSIS")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confidence distribution
ax1 = axes[0, 0]
ax1.hist(df_results['majority_vote_confidence'], bins=20, color='#9B59B6', edgecolor='black', alpha=0.7)
ax1.axvline(df_results['majority_vote_confidence'].mean(), color='red', linestyle='--', 
           label=f'Mean: {df_results["majority_vote_confidence"].mean():.1f}%')
ax1.set_xlabel('Majority Vote Confidence (%)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Majority Vote Confidence', fontsize=14)
ax1.legend()

# 2. Confidence vs Correctness
ax2 = axes[0, 1]
correct_conf = df_results[df_results['is_correct']]['majority_vote_confidence']
incorrect_conf = df_results[~df_results['is_correct']]['majority_vote_confidence']

if len(incorrect_conf) > 0:
    bp = ax2.boxplot([correct_conf.values, incorrect_conf.values], labels=['Correct', 'Incorrect'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ECC71')
    bp['boxes'][1].set_facecolor('#E74C3C')
else:
    bp = ax2.boxplot([correct_conf.values], labels=['Correct'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ECC71')
ax2.set_ylabel('Confidence (%)', fontsize=12)
ax2.set_title('Confidence: Correct vs Incorrect', fontsize=14)

# 3. Scatter: Confidence vs Time
ax3 = axes[1, 0]
colors_scatter = ['#2ECC71' if c else '#E74C3C' for c in df_results['is_correct']]
ax3.scatter(df_results['majority_vote_confidence'], df_results['problem_time_seconds'], 
           c=colors_scatter, alpha=0.7, edgecolors='black', linewidth=0.5)
ax3.set_xlabel('Majority Vote Confidence (%)', fontsize=12)
ax3.set_ylabel('Problem Time (seconds)', fontsize=12)
ax3.set_title('Confidence vs Problem Time (Green=Correct, Red=Incorrect)', fontsize=14)

# 4. Accuracy by confidence bucket
ax4 = axes[1, 1]
df_results['confidence_bucket'] = pd.cut(df_results['majority_vote_confidence'], 
                                          bins=[0, 25, 50, 75, 100], 
                                          labels=['0-25%', '25-50%', '50-75%', '75-100%'])

bucket_stats = df_results.groupby('confidence_bucket', observed=True).agg({
    'is_correct': ['sum', 'count']
}).reset_index()
bucket_stats.columns = ['bucket', 'correct', 'total']
bucket_stats['accuracy'] = bucket_stats['correct'] / bucket_stats['total'] * 100

bars = ax4.bar(bucket_stats['bucket'].astype(str), bucket_stats['accuracy'], 
              color='#3498DB', edgecolor='black')
ax4.set_xlabel('Confidence Bucket', fontsize=12)
ax4.set_ylabel('Accuracy (%)', fontsize=12)
ax4.set_title('Accuracy by Confidence Level', fontsize=14)
ax4.set_ylim(0, 105)

for bar, acc in zip(bars, bucket_stats['accuracy']):
    ax4.annotate(f'{acc:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, acc),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'confidence_analysis_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print confidence statistics
print("\n📊 CONFIDENCE STATISTICS:")
print(f"\n   Overall:")
print(f"   • Mean Confidence: {df_results['majority_vote_confidence'].mean():.2f}%")
print(f"   • Median Confidence: {df_results['majority_vote_confidence'].median():.2f}%")
print(f"   • Min Confidence: {df_results['majority_vote_confidence'].min():.2f}%")
print(f"   • Max Confidence: {df_results['majority_vote_confidence'].max():.2f}%")

if len(correct_conf) > 0:
    print(f"\n   Correct Problems:")
    print(f"   • Mean Confidence: {correct_conf.mean():.2f}%")

if len(incorrect_conf) > 0:
    print(f"\n   Incorrect Problems:")
    print(f"   • Mean Confidence: {incorrect_conf.mean():.2f}%")

In [ ]:
# ============ TIME ANALYSIS ============
print("=" * 70)
print("⏱️ TIME ANALYSIS")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time distribution histogram
ax1 = axes[0, 0]
ax1.hist(df_results['problem_time_seconds'], bins=20, color='#1ABC9C', edgecolor='black', alpha=0.7)
ax1.axvline(df_results['problem_time_seconds'].mean(), color='red', linestyle='--', 
           label=f'Mean: {df_results["problem_time_seconds"].mean():.1f}s')
ax1.axvline(df_results['problem_time_seconds'].median(), color='blue', linestyle=':', 
           label=f'Median: {df_results["problem_time_seconds"].median():.1f}s')
ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Problem Solving Time', fontsize=14)
ax1.legend()

# 2. Time vs Correctness
ax2 = axes[0, 1]
correct_times = df_results[df_results['is_correct']]['problem_time_seconds']
incorrect_times = df_results[~df_results['is_correct']]['problem_time_seconds']

if len(incorrect_times) > 0:
    bp = ax2.boxplot([correct_times.values, incorrect_times.values], labels=['Correct', 'Incorrect'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ECC71')
    bp['boxes'][1].set_facecolor('#E74C3C')
else:
    bp = ax2.boxplot([correct_times.values], labels=['Correct'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ECC71')
ax2.set_ylabel('Time (seconds)', fontsize=12)
ax2.set_title('Time: Correct vs Incorrect Problems', fontsize=14)

# 3. Time progression across problems
ax3 = axes[1, 0]
ax3.plot(range(len(df_results)), df_results['problem_time_seconds'].values, 
        marker='o', color='#3498DB', linewidth=1, markersize=4)
ax3.fill_between(range(len(df_results)), df_results['problem_time_seconds'].values, alpha=0.3)
ax3.set_xlabel('Problem Index (Order Solved)', fontsize=12)
ax3.set_ylabel('Time (seconds)', fontsize=12)
ax3.set_title('Problem Solving Time Progression', fontsize=14)

# Add trend line
z = np.polyfit(range(len(df_results)), df_results['problem_time_seconds'].values, 1)
p = np.poly1d(z)
ax3.plot(range(len(df_results)), p(range(len(df_results))), "r--", alpha=0.8, label='Trend')
ax3.legend()

# 4. Budget utilization
ax4 = axes[1, 1]
budget_used_pct = (df_results['problem_time_seconds'] / df_results['budget_seconds'] * 100).values
ax4.hist(budget_used_pct, bins=20, color='#F39C12', edgecolor='black', alpha=0.7)
ax4.axvline(np.mean(budget_used_pct), color='red', linestyle='--', 
           label=f'Mean: {np.mean(budget_used_pct):.1f}%')
ax4.set_xlabel('Budget Utilization (%)', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Budget Utilization Distribution', fontsize=14)
ax4.legend()

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'time_analysis_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print time statistics
print("\n📊 TIME STATISTICS BY CORRECTNESS:")
if len(correct_times) > 0:
    print(f"\n   Correct Problems:")
    print(f"   • Mean Time: {correct_times.mean():.2f}s")
    print(f"   • Median Time: {correct_times.median():.2f}s")

if len(incorrect_times) > 0:
    print(f"\n   Incorrect Problems:")
    print(f"   • Mean Time: {incorrect_times.mean():.2f}s")
    print(f"   • Median Time: {incorrect_times.median():.2f}s")

# Time by hardness
print("\n📊 TIME STATISTICS BY HARDNESS:")
if len(df_hard) > 0:
    print(f"\n   HARD Problems:")
    print(f"   • Mean Time: {df_hard['problem_time_seconds'].mean():.2f}s")
    print(f"   • Median Time: {df_hard['problem_time_seconds'].median():.2f}s")

if len(df_easy) > 0:
    print(f"\n   EASY Problems:")
    print(f"   • Mean Time: {df_easy['problem_time_seconds'].mean():.2f}s")
    print(f"   • Median Time: {df_easy['problem_time_seconds'].median():.2f}s")

In [ ]:
# ============ ATTEMPT ANALYSIS ============
print("=" * 70)
print("🔄 ATTEMPT ANALYSIS")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Successful attempts distribution
ax1 = axes[0]
attempt_dist = df_results['successful_attempts'].value_counts().sort_index()
bars = ax1.bar(attempt_dist.index, attempt_dist.values, color='#3498DB', edgecolor='black')
ax1.set_xlabel('Number of Successful Attempts', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Successful Attempts per Problem', fontsize=14)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

# 2. Success rate by hardness
ax2 = axes[1]
if len(df_hard) > 0 or len(df_easy) > 0:
    categories = []
    success_rates = []
    
    if len(df_hard) > 0:
        hard_success = df_hard['successful_attempts'].sum()
        hard_total = df_hard['total_attempts'].sum()
        categories.append('HARD')
        success_rates.append(hard_success / hard_total * 100 if hard_total > 0 else 0)
    
    if len(df_easy) > 0:
        easy_success = df_easy['successful_attempts'].sum()
        easy_total = df_easy['total_attempts'].sum()
        categories.append('EASY')
        success_rates.append(easy_success / easy_total * 100 if easy_total > 0 else 0)
    
    bars = ax2.bar(categories, success_rates, color=['#E74C3C', '#2ECC71'][:len(categories)], edgecolor='black')
    ax2.set_ylabel('Attempt Success Rate (%)', fontsize=12)
    ax2.set_title('Attempt Success Rate by Hardness', fontsize=14)
    ax2.set_ylim(0, 105)
    
    for bar, rate in zip(bars, success_rates):
        ax2.annotate(f'{rate:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, rate),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'attempt_analysis_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print attempt statistics
print("\n📊 ATTEMPT STATISTICS:")
print(f"   Total Attempts: {df_results['total_attempts'].sum()}")
print(f"   Successful Attempts: {df_results['successful_attempts'].sum()}")
print(f"   Failed Attempts: {df_results['total_attempts'].sum() - df_results['successful_attempts'].sum()}")
print(f"   Overall Attempt Success Rate: {df_results['successful_attempts'].sum() / df_results['total_attempts'].sum() * 100:.2f}%")
print(f"   Avg Successful Attempts per Problem: {df_results['successful_attempts'].mean():.2f}")

In [ ]:
# ============ ANSWER DISTRIBUTION ANALYSIS ============
print("=" * 70)
print("📝 ANSWER DISTRIBUTION ANALYSIS")
print("=" * 70)

# Analyze answer distributions per problem
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Number of unique answers per problem
unique_answers_per_problem = []
for _, row in df_results.iterrows():
    dist = row['answer_distribution']
    if isinstance(dist, str):
        dist = json.loads(dist.replace("'", '"'))
    unique_answers_per_problem.append(len(dist) if dist else 0)

ax1 = axes[0]
unique_counts = pd.Series(unique_answers_per_problem).value_counts().sort_index()
bars = ax1.bar(unique_counts.index, unique_counts.values, color='#9B59B6', edgecolor='black')
ax1.set_xlabel('Number of Unique Answers', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Unique Answers per Problem', fontsize=14)

for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

# 2. Correlation between unique answers and correctness
ax2 = axes[1]
df_results['unique_answers'] = unique_answers_per_problem
unique_correct = df_results.groupby('unique_answers').agg({
    'is_correct': ['sum', 'count']
}).reset_index()
unique_correct.columns = ['unique_answers', 'correct', 'total']
unique_correct['accuracy'] = unique_correct['correct'] / unique_correct['total'] * 100

bars = ax2.bar(unique_correct['unique_answers'], unique_correct['accuracy'], color='#1ABC9C', edgecolor='black')
ax2.set_xlabel('Number of Unique Answers', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy by Answer Diversity', fontsize=14)
ax2.set_ylim(0, 105)

for bar, acc in zip(bars, unique_correct['accuracy']):
    ax2.annotate(f'{acc:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, acc),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'answer_distribution_analysis_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print statistics
print("\n📊 ANSWER DIVERSITY STATISTICS:")
print(f"   Avg Unique Answers per Problem: {np.mean(unique_answers_per_problem):.2f}")
print(f"   Max Unique Answers: {max(unique_answers_per_problem)}")
print(f"   Problems with Only 1 Unique Answer: {sum(1 for x in unique_answers_per_problem if x == 1)}")

In [ ]:
# ============ COMPREHENSIVE HEATMAP: PROBLEM RESULTS GRID ============
print("=" * 70)
print("🔥 PROBLEM RESULTS HEATMAP")
print("=" * 70)

# Create a comprehensive heatmap showing all problems
fig, ax = plt.subplots(figsize=(18, 8))

# Prepare data for heatmap
df_sorted = df_results.sort_values('problem_id')
n_problems = len(df_sorted)

# Create grid data
grid_data = []
labels = []

# Row 1: Correctness (1 = correct, 0 = incorrect)
grid_data.append(df_sorted['is_correct'].astype(int).values)
labels.append('Correct (1/0)')

# Row 2: Hardness (1 = HARD, 0 = EASY)
grid_data.append([1 if h == 'HARD' else 0 for h in df_sorted['hardness']])
labels.append('Hard (1/0)')

# Row 3: Normalized time (0-1 scale)
time_norm = (df_sorted['problem_time_seconds'] - df_sorted['problem_time_seconds'].min()) / \
            (df_sorted['problem_time_seconds'].max() - df_sorted['problem_time_seconds'].min() + 1e-6)
grid_data.append(time_norm.values)
labels.append('Time (normalized)')

# Row 4: Normalized python calls
calls_norm = (df_sorted['total_python_calls'] - df_sorted['total_python_calls'].min()) / \
             (df_sorted['total_python_calls'].max() - df_sorted['total_python_calls'].min() + 1e-6)
grid_data.append(calls_norm.values)
labels.append('Python Calls (norm)')

# Row 5: Confidence (0-100 scale normalized to 0-1)
grid_data.append((df_sorted['majority_vote_confidence'] / 100).values)
labels.append('Confidence (norm)')

# Row 6: Has errors (1 = has errors, 0 = no errors)
grid_data.append([1 if e > 0 else 0 for e in df_sorted['total_python_errors']])
labels.append('Has Errors (1/0)')

grid_array = np.array(grid_data)

# Create heatmap
im = ax.imshow(grid_array, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

# Set labels
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Problem Index', fontsize=12)
ax.set_title('Problem Results Heatmap\n(Green = Good, Red = Bad for correctness/confidence; normalized for others)', fontsize=14)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Value (0-1 normalized)', fontsize=11)

# Add problem IDs as x-axis labels if not too many
if n_problems <= 50:
    ax.set_xticks(range(n_problems))
    ax.set_xticklabels([str(pid) for pid in df_sorted['problem_id']], rotation=90, fontsize=8)
else:
    # Show every nth label
    step = n_problems // 20
    ax.set_xticks(range(0, n_problems, step))
    ax.set_xticklabels([str(df_sorted['problem_id'].iloc[i]) for i in range(0, n_problems, step)], rotation=90, fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'problem_heatmap_{log_timestamp}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============ FINAL SUMMARY REPORT ============
print("=" * 70)
print("📋 FINAL BENCHMARK SUMMARY REPORT")
print("=" * 70)

report = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    BENCHMARK RESULTS SUMMARY                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 OVERALL PERFORMANCE                                              ║
║  ─────────────────────────────────────────────────────────────────   ║
║  Total Problems:        {len(df_results):>6}                                      ║
║  Correct:               {df_results['is_correct'].sum():>6}                                      ║
║  Incorrect:             {len(df_results) - df_results['is_correct'].sum():>6}                                      ║
║  Accuracy:              {df_results['is_correct'].mean()*100:>6.2f}%                                    ║
║                                                                      ║
║  🎯 BY DIFFICULTY                                                    ║
║  ─────────────────────────────────────────────────────────────────   ║
║  HARD Problems:         {len(df_hard):>6} ({df_hard['is_correct'].sum():>3} correct, {df_hard['is_correct'].mean()*100 if len(df_hard) > 0 else 0:>5.1f}%)           ║
║  EASY Problems:         {len(df_easy):>6} ({df_easy['is_correct'].sum():>3} correct, {df_easy['is_correct'].mean()*100 if len(df_easy) > 0 else 0:>5.1f}%)           ║
║                                                                      ║
║  ⏱️ TIME STATISTICS                                                  ║
║  ─────────────────────────────────────────────────────────────────   ║
║  Total Runtime:         {df_results['problem_time_seconds'].sum():>6.1f}s ({df_results['problem_time_seconds'].sum()/60:.1f} min)               ║
║  Avg per Problem:       {df_results['problem_time_seconds'].mean():>6.1f}s                                    ║
║  Time Saved:            {df_results['time_saved_seconds'].sum():>6.1f}s                                    ║
║                                                                      ║
║  🐍 CODE EXECUTION                                                   ║
║  ─────────────────────────────────────────────────────────────────   ║
║  Total Python Calls:    {df_results['total_python_calls'].sum():>6}                                      ║
║  Total Python Errors:   {df_results['total_python_errors'].sum():>6}                                      ║
║  Python Success Rate:   {(df_results['total_python_calls'].sum() - df_results['total_python_errors'].sum()) / max(1, df_results['total_python_calls'].sum()) * 100:>6.1f}%                                    ║
║                                                                      ║
║  🗳️ MAJORITY VOTE                                                    ║
║  ─────────────────────────────────────────────────────────────────   ║
║  Avg Confidence:        {df_results['majority_vote_confidence'].mean():>6.1f}%                                    ║
║  Min Confidence:        {df_results['majority_vote_confidence'].min():>6.1f}%                                    ║
║  Max Confidence:        {df_results['majority_vote_confidence'].max():>6.1f}%                                    ║
║                                                                      ║
║  📁 OUTPUT FILES                                                     ║
║  ─────────────────────────────────────────────────────────────────   ║
║  Log File: {LOG_FILE:<57} ║
║  Summary: {SUMMARY_FILE:<58} ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""

print(report)

# Export summary as CSV
summary_csv = os.path.join(LOG_DIR, f'benchmark_results_{log_timestamp}.csv')
df_results.to_csv(summary_csv, index=False)
print(f"📁 Results CSV saved to: {summary_csv}")

print("\n✅ Benchmark analysis complete!")